In [ ]:
import sys
import torch

from time import time
import numpy as np

from eszd import ESZD
from eszd.directions import RandomCoords, HouseholderQRDirections, HouseholderReflection

import matplotlib.pyplot as plt

sys.path.append("../")
from synthetic_utils import *


In [ ]:
def run_experiment(optimizer, T = 100, reps = 5):
    f0 = optimizer.target(optimizer.x).item()
    results = [[f0] for _ in range(reps)]
    ctime = [[0.0] for _ in range(reps)]

    for r in range(reps):
        for k in range(T):
            it_time = time()
            x = optimizer.step()
            it_time = time() - it_time
            ctime[r].append(it_time)
            results[r].append(optimizer.target(x).item())
    mu_f, std_f = np.mean(results, axis=0), np.std(results, axis=0)
    ctime = np.cumsum(ctime, axis=1)
    mu_t, std_t = np.mean(ctime, axis=0), np.std(ctime, axis=0)
    return mu_f, std_f, mu_t, std_t


def plot_result(labels, results):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))
    for (lab, ris) in zip(labels, results):
        mu, std, mu_t, std_t = ris
        ax1.plot(range(len(mu)), mu, '-', label=lab)
        ax2.plot(range(len(mu_t)), mu_t, '-', label=lab)
        ax3.plot(mu_t, mu, '-', label=lab)

#    fig.tight_layout()
    ax1.set_xlabel("$k$", fontsize=18)
    ax1.set_ylabel("$f(x_k)$", fontsize=18)    
    ax1.set_yscale('log')
    ax1.legend(fontsize=14)
    ax2.set_xlabel("$k$", fontsize=18)
    ax2.set_ylabel("$s$", fontsize=18)
    ax2.set_yscale('log')
    ax3.set_xlabel("$s$", fontsize=18)
    ax3.set_ylabel("$f(x_k)$", fontsize=18)
    ax3.set_yscale('log')


# Coordinate directions vs Random Structured


In [ ]:
d = 1000
fun_1 = Sphere(d)

In [ ]:
l = 100
dtype = torch.float64
device = 'cpu'

P1 = RandomCoords(d = d, l = l, dtype=dtype, device=device)
P2 = HouseholderQRDirections(d = d, l = l, dtype=dtype, device=device)
P3 = HouseholderReflection(d = d, l = l, dtype = dtype, device = device)
P4 = RandomCoords(d = d, l = l, dct = True, dtype=dtype, device=device)

P1()
P2()
P3()
P4()


In [ ]:

x0 = torch.ones((1, d), dtype=dtype, device=device)

alpha = lambda k : 0.75
h = lambda k : 1e-7


random_coords = ESZD(fun_1, x0.clone(), P = P1, alpha = alpha, h = h, single_point=False) 
qr_directions = ESZD(fun_1, x0.clone(), P = P2, alpha = alpha, h = h, single_point=False) 
reflected = ESZD(fun_1, x0.clone(), P = P3, alpha = alpha, h = h)
dct_dir = ESZD(fun_1, x0.clone(), P = P4, alpha=alpha, h = h)


In [ ]:
reps, T = 10, 100
LABELS = ['random permutation', 'QR', 'random reflection', 'DCT on permutation']

coord_ris = run_experiment(random_coords, T = T, reps = reps)
qr_ris = run_experiment(qr_directions, T = T, reps = reps)
ref_ris = run_experiment(reflected, T = T, reps = reps)
dct_ris = run_experiment(dct_dir, T = T, reps = reps)

results = [coord_ris, qr_ris, ref_ris, dct_ris]


In [ ]:
plot_result(LABELS, results)

### Sparse target

In [ ]:
fun_2 = SparseMatrix(d = d, l_dim=10)

In [ ]:
x0 = torch.ones((1, d), dtype=dtype)

alpha = lambda k : 0.001
h = lambda k : 1e-7


random_coords = ESZD(fun_2, x0.clone(), P = P1, alpha = alpha, h = h, single_point=False) 
qr_directions = ESZD(fun_2, x0.clone(), P = P2, alpha = alpha, h = h, single_point=False) 
reflected = ESZD(fun_2, x0.clone(), P = P3, alpha = alpha, h = h)
dct_dir = ESZD(fun_2, x0.clone(), P = P4, alpha=alpha, h = h)

In [ ]:
reps, T = 10, 100
LABELS = ['random permutation', 'QR', 'random reflection', 'DCT on permutation']

coord_ris = run_experiment(random_coords, T = T, reps = reps)
qr_ris = run_experiment(qr_directions, T = T, reps = reps)
ref_ris = run_experiment(reflected, T = T, reps = reps)
dct_ris = run_experiment(dct_dir, T = T, reps = reps)

results = [coord_ris, qr_ris, ref_ris, dct_ris]


In [ ]:
plot_result(LABELS, results)

### Scalar prod

In [ ]:
fun_3 = ScalarProd(d = d, l_dim=10)